In [5]:
import os
import pandas as pd
from pathlib import Path

imputed_dirs = {
    "EPA": Path("/home/rishi/ML Projects/Air Pollution/EPA/imputed"),
    "AURN": Path("/home/rishi/ML Projects/Air Pollution/AURN/imputed"),
    "CPCB": Path("/home/rishi/ML Projects/Air Pollution/CPCB/sites_imputed"),
    "CNEMC": Path("/home/rishi/ML Projects/Air Pollution/CNEMC/imputed"),
}

all_rows = []
for dataset, imputed_dir in imputed_dirs.items():
    for f in sorted(imputed_dir.glob("*.csv")):
        df = pd.read_csv(f)
        val_col = [c for c in df.columns if c != "Timestamp"][0]
        vals = df[val_col].dropna()
        pollutant = f.stem.rsplit("_", 1)[-1]

        all_rows.append({
            "dataset": dataset,
            "file": f.name,
            "pollutant": pollutant,
            "min": vals.min(),
            "q05": vals.quantile(0.05),
            "q25": vals.quantile(0.25),
            "median": vals.quantile(0.5),
            "q75": vals.quantile(0.75),
            "q95": vals.quantile(0.95),
            "max": vals.max(),
        })

stats = pd.DataFrame(all_rows)

all_summaries = []
for dataset in imputed_dirs:
    ds = stats[stats["dataset"] == dataset]
    summary = ds.groupby("pollutant").agg(
        n_sites=("file", "count"),
        overall_min=("min", "min"),
        overall_max=("max", "max"),
        avg_q00=("min", "mean"),
        avg_q05=("q05", "mean"),
        avg_q25=("q25", "mean"),
        avg_q50=("median", "mean"),
        avg_q75=("q75", "mean"),
        avg_q95=("q95", "mean"),
        avg_q100=("max", "mean"),
    ).round(3)
    summary.insert(0, "dataset", dataset)
    all_summaries.append(summary)
    print(f"\n{'='*60}")
    print(f" {dataset} — {len(ds)} files, {ds['pollutant'].nunique()} pollutants")
    print(f"{'='*60}")
    display(summary)

combined = pd.concat(all_summaries)

# Sites per pollutant per dataset pivot table
site_counts = stats.groupby(["dataset", "pollutant"]).size().unstack(fill_value=0)
site_counts = site_counts.reindex(imputed_dirs.keys())
print(f"\n{'='*60}")
print(" Sites per pollutant per dataset")
print(f"{'='*60}")
display(site_counts)

# Export both tables to Excel
out_path = "/home/rishi/ML Projects/Air Pollution/dataset_statistics.xlsx"
with pd.ExcelWriter(out_path) as writer:
    combined.to_excel(writer, sheet_name="Summary Statistics")
    site_counts.to_excel(writer, sheet_name="Sites Per Pollutant")
print(f"\nExported to dataset_statistics.xlsx")


 EPA — 1667 files, 6 pollutants


,dataset,n_sites,overall_min,overall_max,avg_q00,avg_q05,avg_q25,avg_q50,avg_q75,avg_q95,avg_q100
pollutant,,,,,,,,,,,
CO,EPA,90,0.0,18.016,0.013,0.125,0.198,0.270,0.392,0.704,2.999
NO2,EPA,212,0.0,212.440,0.050,2.897,6.608,11.724,21.049,40.393,96.843
Ozone,EPA,488,0.0,458.640,1.799,18.216,42.286,61.755,80.250,107.840,184.240
PM10,EPA,220,0.0,15130.000,0.118,4.219,10.016,16.506,26.276,54.240,1592.342
PM2.5,EPA,445,0.0,1285.600,0.076,1.561,3.831,6.166,9.504,18.258,246.372
SO2,EPA,212,0.0,2838.508,0.009,0.113,0.543,1.243,2.317,9.929,153.339



 AURN — 259 files, 6 pollutants


,dataset,n_sites,overall_min,overall_max,avg_q00,avg_q05,avg_q25,avg_q50,avg_q75,avg_q95,avg_q100
pollutant,,,,,,,,,,,
CO,AURN,1,0.0,2.585,0.000,0.070,0.116,0.151,0.221,0.431,2.585
NO2,AURN,70,0.0,241.358,0.137,3.396,8.211,14.265,23.358,43.529,116.264
Ozone,AURN,39,0.0,234.395,0.869,13.521,38.166,53.713,66.838,85.777,171.039
PM10,AURN,80,0.0,857.175,0.186,3.483,7.328,11.171,16.802,30.893,253.973
PM2.5,AURN,61,0.0,305.591,0.129,1.681,3.658,5.640,9.051,19.712,118.026
SO2,AURN,8,0.0,410.283,0.000,0.156,0.357,0.576,0.916,3.119,170.881



 CPCB — 1104 files, 6 pollutants


...


 CNEMC — 8893 files, 6 pollutants


,dataset,n_sites,overall_min,overall_max,avg_q00,avg_q05,avg_q25,avg_q50,avg_q75,avg_q95,avg_q100
pollutant,,,,,,,,,,,
CO,CNEMC,1487,0.0,20.7,0.043,0.292,0.447,0.580,0.763,1.181,4.443
NO2,CNEMC,1481,0.0,1028.0,0.310,5.433,11.157,18.316,30.168,55.011,139.085
Ozone,CNEMC,1488,0.0,693.0,0.254,10.848,36.477,61.671,91.733,145.381,268.905
PM10,CNEMC,1481,0.0,20883.0,0.263,13.607,29.110,46.721,73.973,142.091,1761.722
PM2.5,CNEMC,1471,0.0,10000.0,0.145,5.591,13.942,24.007,40.437,84.205,576.488
SO2,CNEMC,1485,0.0,1476.0,0.606,3.201,5.190,7.003,9.621,16.979,187.417



 Sites per pollutant per dataset


pollutant,CO,NO2,Ozone,PM10,PM2.5,SO2
dataset,,,,,,
EPA,90,212,488,220,445,212
AURN,1,70,39,80,61,8
CPCB,179,179,182,191,190,183
CNEMC,1487,1481,1488,1481,1471,1485



Exported to dataset_statistics.xlsx


In [2]:
# For EPA: check if max values exist in the original hourly files
epa_type_codes = {
    "CO": "42101",
    "SO2": "42401",
    "NO2": "42602",
    "Ozone": "44201",
    "PM10": "81102",
    "PM2.5": "88101",
}
# Conversion factors applied during preprocessing (raw * factor = imputed)
conversion_factors = {
    "SO2": 2.62,
    "PM2.5": 1.0,
    "Ozone": 1.96 * 1000,
    "NO2": 1.88,
    "CO": 1.15,
    "PM10": 1.0,
}
raw_dir = Path("/home/rishi/ML Projects/Air Pollution/EPA/unzip_dir")

epa_stats = stats[stats["dataset"] == "EPA"]
max_per_pol = epa_stats.loc[epa_stats.groupby("pollutant")["max"].idxmax()]

results = []
for _, row in max_per_pol.iterrows():
    pol = row["pollutant"]
    fname = row["file"]
    max_val = row["max"]
    factor = conversion_factors[pol]

    # Parse site info from filename: site_STATE_COUNTY_SITE_POLLUTANT.csv
    parts = fname.replace(".csv", "").split("_")
    state, county, site_num = parts[1], parts[2], parts[3]

    # Read imputed file to get the timestamp of the max
    imp_df = pd.read_csv(Path("/home/rishi/ML Projects/Air Pollution/EPA/imputed") / fname)
    val_col = [c for c in imp_df.columns if c != "Timestamp"][0]
    max_idx = imp_df[val_col].idxmax()
    max_ts = pd.Timestamp(imp_df.loc[max_idx, "Timestamp"])

    # Find the right raw file by year
    type_code = epa_type_codes[pol]
    year = max_ts.year
    raw_file = raw_dir / f"hourly_{type_code}_{year}.csv"

    found = False
    raw_value = None
    raw_converted = None
    if raw_file.exists():
        raw_df = pd.read_csv(raw_file, low_memory=False)
        match = raw_df[
            (raw_df["State Code"].astype(str) == state) &
            (raw_df["County Code"].astype(str) == county) &
            (raw_df["Site Num"].astype(str) == site_num)
        ]
        if not match.empty:
            raw_max = match["Sample Measurement"].max()
            raw_value = raw_max
            raw_converted = raw_max * factor
            # Check if the imputed max matches a converted raw value
            found = any(((match["Sample Measurement"] * factor) - max_val).abs() < 0.01)

    results.append({
        "pollutant": pol,
        "file": fname,
        "max_timestamp": str(max_ts),
        "imputed_max": round(max_val, 3),
        "raw_max": raw_value,
        "raw_max_converted": round(raw_converted, 3) if raw_converted is not None else None,
        "factor": factor,
        "found_in_raw": found,
    })

results_df = pd.DataFrame(results)
print("EPA — Max value verification against raw hourly files:")
display(results_df)

# For CPCB: check if max values exist in the pre-imputation separated files
# CPCB has no unit conversion — raw and imputed use the same units
cpcb_raw_dir = Path("/home/rishi/ML Projects/Air Pollution/CPCB/separated")

cpcb_stats = stats[stats["dataset"] == "CPCB"]
cpcb_max_per_pol = cpcb_stats.loc[cpcb_stats.groupby("pollutant")["max"].idxmax()]

cpcb_results = []
for _, row in cpcb_max_per_pol.iterrows():
    pol = row["pollutant"]
    fname = row["file"]
    max_val = row["max"]

    # Read imputed file to get the timestamp of the max
    imp_df = pd.read_csv(Path("/home/rishi/ML Projects/Air Pollution/CPCB/sites_imputed") / fname)
    val_col = [c for c in imp_df.columns if c != "Timestamp"][0]
    max_idx = imp_df[val_col].idxmax()
    max_ts = pd.Timestamp(imp_df.loc[max_idx, "Timestamp"])

    # Check the corresponding pre-imputation separated file
    raw_file = cpcb_raw_dir / fname
    found = False
    raw_max = None
    if raw_file.exists():
        raw_df = pd.read_csv(raw_file)
        raw_val_col = [c for c in raw_df.columns if c != "Timestamp"][0]
        raw_vals = raw_df[raw_val_col].dropna()
        raw_max = raw_vals.max()
        found = any((raw_df[raw_val_col].dropna() - max_val).abs() < 0.01)

    cpcb_results.append({
        "pollutant": pol,
        "file": fname,
        "max_timestamp": str(max_ts),
        "imputed_max": round(max_val, 3),
        "raw_max": raw_max,
        "found_in_raw": found,
    })

cpcb_results_df = pd.DataFrame(cpcb_results)
print("\nCPCB — Max value verification against pre-imputation separated files:")
display(cpcb_results_df)

# For CNEMC: check if max values exist in the pre-imputation separated files
# CNEMC has no unit conversion — raw and imputed use the same units
cnemc_raw_dir = Path("/home/rishi/ML Projects/Air Pollution/CNEMC/separated")

cnemc_stats = stats[stats["dataset"] == "CNEMC"]
cnemc_max_per_pol = cnemc_stats.loc[cnemc_stats.groupby("pollutant")["max"].idxmax()]

cnemc_results = []
for _, row in cnemc_max_per_pol.iterrows():
    pol = row["pollutant"]
    fname = row["file"]
    max_val = row["max"]

    # Read imputed file to get the timestamp of the max
    imp_df = pd.read_csv(Path("/home/rishi/ML Projects/Air Pollution/CNEMC/imputed") / fname)
    val_col = [c for c in imp_df.columns if c != "Timestamp"][0]
    max_idx = imp_df[val_col].idxmax()
    max_ts = pd.Timestamp(imp_df.loc[max_idx, "Timestamp"])

    # Check the corresponding pre-imputation separated file
    raw_file = cnemc_raw_dir / fname
    found = False
    raw_max = None
    if raw_file.exists():
        raw_df = pd.read_csv(raw_file)
        raw_val_col = [c for c in raw_df.columns if c != "Timestamp"][0]
        raw_vals = raw_df[raw_val_col].dropna()
        raw_max = raw_vals.max()
        found = any((raw_df[raw_val_col].dropna() - max_val).abs() < 0.01)

    cnemc_results.append({
        "pollutant": pol,
        "file": fname,
        "max_timestamp": str(max_ts),
        "imputed_max": round(max_val, 3),
        "raw_max": raw_max,
        "found_in_raw": found,
    })

cnemc_results_df = pd.DataFrame(cnemc_results)
print("\nCNEMC — Max value verification against pre-imputation separated files:")
display(cnemc_results_df)

EPA — Max value verification against raw hourly files:


,pollutant,file,max_timestamp,imputed_max,raw_max,raw_max_converted,factor,found_in_raw
0,CO,site_42_3_1301_CO.csv,2024-07-09 01:00:00,18.016,15.666,18.016,1.15,True
1,NO2,site_49_21_5_NO2.csv,2023-03-06 06:00:00,212.440,113.000,212.440,1.88,True
2,Ozone,site_12_5_6_Ozone.csv,2024-12-06 16:00:00,458.640,0.234,458.640,1960.00,True
3,PM10,site_4_21_7004_PM10.csv,2022-08-14 17:00:00,15130.000,15130.000,15130.000,1.00,True
4,PM2.5,site_35_13_22_PM2.5.csv,2025-03-06 15:00:00,1285.600,1285.600,1285.600,1.00,True
5,SO2,site_15_1_2020_SO2.csv,2023-09-11 03:00:00,2838.508,1083.400,2838.508,2.62,True



CPCB — Max value verification against pre-imputation separated files:


,pollutant,file,max_timestamp,imputed_max,raw_max,found_in_raw
0,CO,site_105_North_Campus_DU_Delhi_IMD_CO.csv,2023-02-28 16:00:00,41.055,47.810,True
1,NO2,site_301_Anand_Vihar_Delhi_DPCC_NO2.csv,2022-11-10 18:00:00,499.200,499.200,True
2,Ozone,site_5587_Bardowali_Agartala_Tripura_SPCB_Ozon...,2025-04-02 08:00:00,497.085,497.085,True
3,PM10,site_118_DTU_Delhi_CPCB_PM10.csv,2022-07-02 18:00:00,1000.000,1000.000,True
4,PM2.5,site_118_DTU_Delhi_CPCB_PM2.5.csv,2023-11-22 14:00:00,1000.000,1000.000,True
5,SO2,site_118_DTU_Delhi_CPCB_SO2.csv,2023-04-13 23:00:00,199.900,199.900,True



CNEMC — Max value verification against pre-imputation separated files:


,pollutant,file,max_timestamp,imputed_max,raw_max,found_in_raw
0,CO,site_3615A_CO.csv,2023-12-28 11:00:00,20.7,20.7,True
1,NO2,site_2671A_NO2.csv,2023-05-03 22:00:00,1028.0,1028.0,True
2,Ozone,site_1718A_Ozone.csv,2023-07-10 22:00:00,693.0,693.0,True
3,PM10,site_2708A_PM10.csv,2025-04-23 12:00:00,20883.0,20883.0,True
4,PM2.5,site_1304A_PM2.5.csv,2024-02-09 21:00:00,10000.0,10000.0,True
5,SO2,site_3248A_SO2.csv,2025-05-06 11:00:00,1476.0,1476.0,True
